<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-1-Lab-2/Unit1_Lab2_Tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Breast Cancer Classification Demo – Decision Tree Walkthrough
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc

In [ ]:
# STEP 1: Load and prepare the dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # 0 = malignant, 1 = benign
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})  # Add human-readable label

In [ ]:
# STEP 2: Exploratory Data Analysis (EDA)

# 2a. Show shape and class distribution
print("✅ Dataset shape:", df.shape)
print("✅ Class distribution:\n", df['diagnosis'].value_counts())

In [ ]:
# 2b. Show top 10 features most correlated with diagnosis
correlations = df.corr(numeric_only=True)['target'].sort_values(ascending=False)
print("\n✅ Top 10 correlated features with 'target':\n", correlations[1:11])  # skip target itself

In [ ]:
# 2c. Visualize distribution of a strong predictive feature
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='worst perimeter', hue='diagnosis', bins=30, kde=True)
plt.title("Distribution of 'Worst Perimeter' by Diagnosis")
plt.xlabel('Worst Perimeter')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# 2d. Heatmap of top 10 predictive features
top_features = correlations[1:11].index.tolist()
plt.figure(figsize=(10, 8))
sns.heatmap(df[top_features + ['target']].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap: Top 10 Predictive Features")
plt.tight_layout()
plt.show()

In [ ]:
# 2e. Summary statistics (first 10 features)
print("\n📊 Summary statistics for first 10 features:\n")
print(df.describe().T.head(10))  # Transpose for readability# 2f. Check for missing values
missing_values = df.isnull().sum()
missing_total = missing_values.sum()
if missing_total == 0:
    print("\n✅ No missing values in the dataset.")
else:
    print("\n⚠️ Missing values found:\n", missing_values[missing_values > 0])

In [ ]:
# STEP 3: Prepare features and target
X = df.drop(columns=['target', 'diagnosis'])  # Remove label columns
y = df['target']

In [ ]:
# STEP 4: Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
# STEP 5: Train a Decision Tree Classifier
clf = DecisionTreeClassifier(criterion='entropy', random_state=100, max_depth=4, min_samples_leaf=5)
clf.fit(X_train, y_train)

In [ ]:
# STEP 6: Make predictions
y_pred = clf.predict(X_test)
y_probs = clf.predict_proba(X_test)[:, 1]  # Probability scores for ROC

In [ ]:
# STEP 7: Evaluate model performance
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n📈 Evaluation Metrics:")
print("✅ Accuracy:", accuracy)
print("✅ Precision:", precision)
print("✅ Recall:", recall)
print("✅ F1 Score:", f1)

In [ ]:
# STEP 8: Plot ROC Curve and calculate AUC
def plot_roc(y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label='AUC = %.2f' % roc_auc, color='darkorange')
    plt.plot([0, 1], [0, 1], 'b--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return roc_auc

dtree_auc = plot_roc(y_test, y_probs)
print("✅ AUC Score:", dtree_auc)

In [ ]:
# STEP 9: Stratified K-Fold Cross-Validation
best_accuracy = 0
best_k_fold = 0

print("\n🔁 Cross-Validation Results:")
for k in range(2, 11):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=100)
    scores = cross_val_score(clf, X, y, cv=skf)
    mean_score = scores.mean()

    print(f"k={k}, Cross-Validated Accuracy={mean_score:.4f}")

    if mean_score > best_accuracy:
        best_accuracy = mean_score
        best_k_fold = k

print("✅ Best Cross-Validated Accuracy:", best_accuracy)
print("✅ Best k-Fold:", best_k_fold)

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Visualize the trained Decision Tree with split values
plt.figure(figsize=(18, 8))
plot_tree(clf,
          feature_names=X.columns,
          class_names=['No Disease', 'Disease'],
          filled=True,
          rounded=True,
          precision=2)
plt.title("Decision Tree with Cutoff Scores")
plt.show()

In [ ]:
import seaborn as sns

# Create a feature importance plot
importances = pd.Series(clf.feature_importances_, index=X.columns)
plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Feature Importances from Decision Tree')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.show()